# AI 기반 문장 생성 기술 분석 보고서


## 1. 학생 정보


- 훈련생 고유번호 : P141

- 성명 : 김민지



---

# 2. 실습 파일 분석


# Practice_0 분석


## 2-1. 파일 목적 및 역할


해당 파일이 해결하려는 문제와
구현하려는 기능을 설명하세요.



작성:

practice_0을 처음 열고 확인을 했을 때는 단순히 프롬프트를 넣고 답을 받는 코드인 줄 알았는데, 실제로 따라가 보니 "Attention Is All You Need" 논문 PDF를 지식 근거로 삼아 GPT-4o가 답하도록 만든 RAG(Retrieval Augmented Generation) 시스템이라는 걸 알게 됐다.

- **해결하려는 문제**: LLM은 정보 중에 모르는게 있으면 사실이 아닌 내용을 그럴듯하게 지어내는 환각(hallucination) 문제가 있다. 따라서 현재는 특정 문서(논문)의 내용을 정확히 근거로 답하기 어렵다.
- **구현하려는 기능**: 논문에서 질문과 관련된 부분을 먼저 검색한 뒤 그 근거로 답하게 만들어 환각을 줄인다. 여기에 CoT -> Self-Consistency -> ReAct 흐름의 프롬프트를 얹어 답변이 단계적으로 깊어지도록 유도한다.




---


## 2-2. 적용 기술 분석


해당 파일에서 사용된 AI 기술,
모델,
라이브러리,
데이터 처리 방법을 분석하여 작성하세요.



작성:

코드를 뜯어보며 어떤 기술이 쓰였는지 하나씩 정리해봤다.

| 구분 | 사용 기술 | 역할 |
|---|---|---|
| 프레임워크 | LangChain | RAG 파이프라인 구성 |
| 문서 로딩 | pypdf (Pdf Reader) | 논문 PDF에서 텍스트 추출 |
| 문서 분할 | RecursiveCharacterTextSplitter | 긴 논문을 청크(chunk)로 분할 |
| 임베딩 | OpenAI Embeddings | 텍스트를 의미 벡터로 변환 |
| 벡터 DB | FAISS | 벡터 저장 및 의미 기반 유사도 검색 |
| 생성 모델 | ChatOpenAI (GPT-4o) | 검색 근거 기반 답변 생성 |
| 프롬프트 기법 | CoT → SC → ReAct | 단계적 추론, 정합성 검증, 시뮬레이션 유도 |

데이터 처리는 **PDF -> 텍스트 추출 -> 청크 분할(chunk_size/chunk_overlap) -> 임베딩 벡터화 -> FAISS 저장 -> 질문과 유사한 청크 검색 -> 프롬프트 결합** 순서로 진행된다.
텍스트를 숫자 벡터로 바꿔 텍스트 간의 의미가 가까운 것을 찾는다는 개념을 여기서 처음으로 감 잡을 수 있었다.

위 과정은 RAG를 구현하기 위한 데이터 인덱싱 및 검색 과정에 해당한다.
각 단계에서 데이터의 형태가 어떻게 바뀌고 왜 그렇게 처리하는지를 분석하면 다음과 같다.

- 텍스트 추출: 논문 PDF에서 사람이 읽는 텍스트를 뽑아낸다. 모델이 다루려면 먼저 순수 텍스트 형태여야 하기 때문이다.
- 청크 분할: 긴 논문을 일정 길이의 조각(chunk)으로 나눈다. 논문 전체를 통째로 넣으면 문맥이 너무 길어 검색/입력이 어렵기에 검색 단위를 잘게 나누는 것이다. 이때 chunk_overlap으로 조각끼리 일부를 겹쳐, 경계에서 문맥이 끊기는 것을 막는다.
- 임베딩 벡터화: 각 청크를 의미를 담은 숫자 벡터로 변환한다. 텍스트를 벡터로 바꾸면 "의미가 가까운 정도"를 수치(거리)로 계산할 수 있게 된다.
- FAISS 저장: 이 벡터들을 벡터 DB에 저장해, 질문이 들어왔을 때 의미가 가까운 청크를 빠르게 찾을 수 있게 한다.
- 유사 청크 검색 -> 프롬프트 결합: 질문도 같은 방식으로 벡터화한 뒤, 벡터 거리가 가까운 상위 청크(top_k=3)를 검색해 프롬프트 앞에 근거로 붙인다. 이렇게 하면 모델이 지어내지 않고 검색된 근거를 바탕으로 답하게 된다.

즉 이 파일의 데이터 처리는 "텍스트 -> 청크 -> 벡터 -> 검색 -> 근거 결합" 이라는 흐름으로 단순 문자열 매칭이 아니라 의미 기반 검색을 가능하게 하는 것이 핵심이다.
실제 실행해보니 논문이 89개 청크로 분할되어 FAISS 벡터 DB에 저장됐으며 세 개의 복합 프롬프트 모두 논문 근거 기반의 답변이 생성되는 것을 확인했다.


---


## 2-3. 전체 동작 과정


코드 실행 흐름을 단계별로 설명하세요.



작성:

```
[1 사전 준비 - 질문 전에 1번만 실행]
논문 PDF 입력 (paper-attention.pdf)
        ↓
텍스트 추출 (pypdf)
        ↓
문서 청크 분할 (Text Splitter) — 89개 청크 생성
        ↓
임베딩 변환 + FAISS 벡터 DB 생성

[2 질의응답 - 질문마다 반복 실행]
사용자 질문 입력 (CoT/SC/ReAct 복합 프롬프트)
        ↓
질문과 의미가 가까운 청크 검색 (similarity_search, top_k=3)
        ↓
검색 근거 + 질문을 하나의 프롬프트로 결합
        ↓
GPT-4o가 근거 기반 답변 생성
        ↓
결과 출력
```

- 1 사전 준비 단계: 논문을 검색 가능한 벡터 DB로 만들어 두는 과정으로 프로그램 실행 시 처음 한 번만 수행된다. PDF를 텍스트로 바꾸고, 청크로 나눈 뒤, 각 청크를 벡터로 변환해 FAISS에 저장한다.
- 2 질의응답 단계: 준비된 벡터 DB를 이용해 실제 질문에 답하는 과정으로 질문이 들어올 때마다 반복된다. 세 개의 복합 프롬프트를 순서대로 넣으면, 각 질문마다 관련 청크를 검색해 근거로 붙이고 GPT-4o가 답변을 생성한다.
따라서 "한 번 인덱싱 -> 여러 번 검색·생성" 구조가 RAG 실행의 핵심 흐름이다.




---


## 2-4. 주요 코드 분석


핵심 코드 일부를 작성하고
각 코드가 수행하는 역할을 설명하세요.



### 주요 코드


```python


# 1) 논문 PDF에서 텍스트 추출
from pypdf import PdfReader
def load_paper(path):
    reader = PdfReader(path)
    return "\n".join([p.extract_text() for p in reader.pages if p.extract_text()])
paper_text = load_paper("paper-attention.pdf")

# 2) 검색 기반 답변 생성 함수 (RAG 핵심)
def answer_with_context(prompt, top_k=3):
    related_docs = vectorstore.similarity_search(prompt, k=top_k)   # 관련 문단 검색
    context = "\n\n".join([doc.page_content for doc in related_docs])
    full_prompt = ("너는 논문 기반 연구 보조 AI야.\n"
                   f"다음은 논문에서 발췌한 내용이야:\n\n{context}\n\n"
                   f"이제 아래 복합 질문에 단계적으로 답변해줘:\n{prompt}")  # 근거 + 질문 결합
    llm = ChatOpenAI(model="gpt-4o", temperature=0.3)
    return llm.invoke([HumanMessage(content=full_prompt)]).content  # GPT-4o 답변
```

**각 코드가 수행하는 역할**

- `load_paper`: PDF 모든 페이지에서 텍스트를 뽑아 하나의 문자열로 합친다.
- `similarity_search`: 질문과 의미가 가까운 논문 문단 3개를 FAISS에서 찾아준다. 단순 키워드 매칭이 아니라 임베딩 기반 의미 검색을 한다.
- `full_prompt`: 검색한 근거를 질문 앞에 붙여서 모델이 지어내지 않고 근거를 보고 답하게 만든다.
- `llm.invoke`: 최종적으로 GPT-4o가 답변을 생성한다.




# Practice_1-1 분석


## 3-1. 파일 목적 및 역할


해당 파일이 해결하려는 문제와
구현하려는 기능을 설명하세요.



작성:

practice_1-1에서는 PyTorch로 문자 단위 LSTM을 직접 만들어 다음 글자를 예측하는 모델을 구현한 것이다.

- **해결하려는 문제**: 순서가 있는 텍스트에서 "다음에 올 글자"를 예측하는 언어 모델을 딥러닝으로 직접 구현한다.
- **구현하려는 기능**: "hello world machine learning is fun " 문장 하나를 글자 단위로 학습시켜서 시작 문자열("hello") 뒤를 이어 새 문장을 생성한다.




---


## 3-2. 적용 기술 분석


해당 파일에서 사용된 모델 구조와
문장 생성 방식에 사용된 기술을 분석하세요.



작성:

이 파일은 앞의 RAG와 달리 **모델을 직접 처음부터 학습**시킨다는 점이 가장 큰 차이였다.

- **모델 구조**: Embedding -> LSTM -> Linear 로 이어지는 순환 신경망 (RNN 계열)
- **문장 생성 방식**: 자기회귀(Autoregressive) 방식, 방금 만든 글자를 다시 입력으로 넣어 다음 글자를 예측
- **핵심 기술**: `nn.Embedding`(글자 -> 벡터), `nn.LSTM`(순차 처리/기억), `nn.Linear`(다음 글자 점수)
- **학습**: CrossEntropyLoss(다중 분류 손실) + Adam(최적화)으로 200 epoch 반복
- **선택 방식**: `argmax`로 가장 확률 높은 글자 선택

학습을 돌려보니 loss가 2.8507(epoch 0)에서 0.0802(epoch 180)까지 줄어드는 걸 눈으로 확인할 수 있었는데, 이게 "모델이 점점 배운다"는 뜻이다.




---


## 3-3. 데이터 처리 과정 설명


입력 데이터가 처리되어
최종 문장이 생성되는 과정을 단계별로 설명하세요.



작성:

```
[1 데이터 준비]
문자열 입력 ("hello world machine learning is fun ")
        ↓
고유 문자 추출 & 숫자 인코딩 (char_to_idx / idx_to_char)
        ↓
seq_length(10) 단위로 입력/정답 쌍 생성 (정답은 한 글자 뒤로 밀림)
        ↓
PyTorch Tensor로 변환

[2 학습]
LSTM에 입력 → 다음 글자 예측 → 오차(loss) 계산 → 가중치 수정
        ↓
200 epoch 반복 (loss가 점점 감소)

[3 생성]
시작 문자열("hello") 입력 → 다음 글자 예측(argmax)
        ↓
예측한 글자를 다시 입력으로 넣어 반복 (자기회귀)
        ↓
문장 생성
```

- 1 준비: 모델은 글자를 직접 이해하지 못하기 때문에 먼저 글자를 숫자 ID로 바꾼다(=인코딩). 그리고 "입력 10글자 -> 정답은 한 칸 민 10글자" 쌍을 만드는데, 이는 각 위치에서 바로 다음 글자를 맞히도록 학습시키기 위한 형태다.
- 2 학습: 예측 -> 오차 -> 가중치 수정을 200번 반복하며 모델이 문자 패턴을 익힌다. 이 과정에서 데이터는 숫자(ID)에서 임베딩 벡터로 바뀌어 LSTM 내부를 통과한다.
- 3 생성: 학습이 끝난 모델에 시작 글자를 주면, 다음 글자를 예측하고 그 글자를 다시 입력에 이어 붙이는 식으로 한 글자씩 문장을 완성한다.
즉 데이터는 "글자 -> 숫자 -> 벡터 -> 다음 글자 예측 -> 문자열" 로 변형되며 흐른다.



---


## 3-4. 모델 구조 설명


사용된 모델의 내부 구조와
각 구성 요소의 역할을 설명하세요.



작성:

이 모델은 Embedding -> LSTM -> Linear 세 층이 순서대로 연결된 구조다. 입력한 글자가 각 층을 지나며 다음과 같이 형태가 바뀐다.

모델을 구성하는 세 요소가 각각 무슨 일을 하는지 정리해봤다.

| 구성 요소 | 역할 | 데이터 변화 |
|---|---|---|
| `nn.Embedding(input_size, hidden_size)` | 글자 ID를 의미를 담은 벡터로 변환 | 숫자 1개 → 50차원 벡터 |
| `nn.LSTM(hidden_size, hidden_size, num_layers=2, batch_first=True)` | 글자를 순서대로 처리하며 hidden state(기억 공간, 50칸)에 문맥을 요약해 저장 | 50차원 벡터 → 문맥 반영된 50차원 벡터|
| `nn.Linear(hidden_size, input_size)` | LSTM 출력을 글자 종류 수만큼의 점수로 변환 → 가장 높은 점수가 예측 글자 | 50차원 → 글자 종류 수(점수) |

- 왜 이 구조인가: Embedding은 "글자를 숫자 의미로", LSTM은 "앞 글자들의 문맥을 기억", Linear는 "다음 글자를 고를 점수로" 변환하는 역할을 각각 맡는다. 세 층이 이어져야 "문맥을 보고 다음 글자를 예측"하는 흐름이 완성된다.
- 핵심 하이퍼파라미터: hidden_size=50은 LSTM 내부 기억 공간의 크기, num_layers=2는 LSTM을 2층으로 쌓아 더 복잡한 패턴을 학습하게 하는 설정이다.




---


## 3-5. 주요 코드 분석


문장 생성과 관련된 핵심 코드를 작성하고
코드의 역할을 설명하세요.



### 주요 코드


```python


class CharLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(input_size, hidden_size)   # 글자 -> 벡터
        self.lstm = nn.LSTM(hidden_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, input_size)             # 벡터 -> 글자 점수
    def forward(self, x, hidden):
        x = self.embedding(x)
        out, hidden = self.lstm(x, hidden)
        out = self.fc(out)
        return out, hidden

# 학습 루프 (딥러닝 기본 5단계)
for epoch in range(200):
    output, hidden = model(input_data, None)          # 1) 예측
    loss = criterion(output.view(-1, input_size), target_data.view(-1))  # 2) 오차
    optimizer.zero_grad()                              # 3) 기울기 초기화
    loss.backward()                                    # 4) 역전파
    optimizer.step()                                   # 5) 가중치 수정

# 문장 생성 (자기회귀)
last_char_idx = torch.argmax(output[:, -1, :], dim=1).item()  # 최고 확률 글자 선택
```

**코드의 역할**

- `forward`: 글자를 벡터로 바꾼 뒤 LSTM으로 순서대로 처리하고, 다음 글자 점수를 출력한다.
- 학습 루프: 예측 → 오차 → 역전파 → 가중치 수정을 200번 반복한다. 이 5단계가 딥러닝 학습의 기본 사이클이라는 걸 이 코드로 확실히 익혔다.
- 생성: `argmax`는 매번 가장 확률 높은 글자만 뽑기 때문에 같은 입력이면 항상 같은 결과가 나온다. 생성 다양성이 없다는 단점이 코드 구조에서부터 보였다.


# Practice_1-2 분석


## 4-1. 파일 목적 및 역할


해당 파일이 해결하려는 문제와
구현하려는 기능을 설명하세요.



작성:

practice_1-2는 앞의 LSTM처럼 직접 학습시키는 대신에 이미 학습된 GPT-2(Transformer)를 가져다 쓴다는 점이 달랐다.

- **해결하려는 문제**: 직접 실행해보니 KB 없이 "The capital of South Korea is"를 넣었을 때 "...is so huge, that it is almost impossible to travel..." 처럼 수도 이름 없이 엉뚱한 문장을 지어냈다(= 환각, hallucination).
- **구현하려는 기능**: (1) 사전학습 GPT-2로 프롬프트 이어쓰기 생성, (2) 미니 Knowledge Base에서 사실을 검색해 프롬프트에 넣어 환각을 줄인다. 실제로 KB를 붙인 버전은 "The capital of South Korea is Seoul." 이라고 정확히 답했다.




---


## 4-2. 적용 기술 분석


해당 파일에서 사용된 모델,
라이브러리,
데이터 처리 방식,
문장 생성 방식을 분석하세요.



작성:

- **모델**: 사전학습 GPT-2 (`GPT2LMHeadModel`), Transformer 디코더 구조
- **라이브러리**: HuggingFace `transformers`
- **데이터 처리 방식**: `GPT2Tokenizer`로 문장 ↔ 토큰(단어 조각) 숫자 변환 → generate → 디코딩
- **문장 생성 방식**: 자기회귀 + 샘플링(`do_sample=True`, `temperature=0.7`)으로 다양한 문장 생성
- **지식 결합**: `knowledge_base`(미니 사전) + `retrieve_fact`로 사실을 검색해 프롬프트에 주입 (RAG의 기본 아이디어)

1-1의 LSTM은 글자 단위였는데 여기선 토큰 단위라는 점, 그리고 순서대로가 아니라 Self-Attention으로 문장 전체를 한 번에 본다는 점이 가장 큰 차이라고 느꼈다.




---


## 4-3. 입력부터 결과 생성 과정


사용자의 입력이 최종 문장으로 생성되는
전체 과정을 단계별로 설명하세요.



작성:

```
[기본 생성]
입력 문장 ("The capital of South Korea is")
        ↓  토큰화 (Tokenizer)
        ↓  GPT-2 generate (자기회귀로 다음 토큰 예측 반복)
        ↓  디코딩 (토큰 -> 문장)
        ↓  출력 결과

[지식 결합(RAG 스타일) 생성]
질문 입력 ("What is the capital of South Korea?")
        ↓  retrieve_fact: knowledge_base에서 관련 사실 검색
        ↓  "사실 + 질문"을 하나의 프롬프트로 결합
        ↓  GPT-2 생성
        ↓  환각이 줄어든 답변 출력
```

- 기본 생성: 입력 문장을 토큰으로 바꿔 GPT-2에 넣으면, 모델이 다음 토큰을 하나씩 예측해 이어 붙인다. 사전학습 지식에만 의존하므로 사실이 틀릴 수 있다(환각).
- 지식 결합 생성: 생성 전에 retrieve_fact로 사실을 먼저 찾아 프롬프트 앞에 붙인다. 모델이 그 사실을 보고 답하므로 정확도가 올라간다. 두 방식의 코드 흐름은 같고, "사실을 미리 넣어주느냐" 한 단계만 다르다.
- 즉 입력은 "문장 → 토큰 → (사실 결합) → 생성 → 문장" 순으로 처리되며, 사실 결합 단계가 있고 없고에 따라 결과 품질이 달라진다.

---


## 4-4. 모델 구조 설명


사용된 모델 구조와
각 구성 요소의 역할을 설명하세요.



작성:

이 파일은 모델을 직접 만들지 않고 사전학습된 GPT-2(Transformer 디코더)를 가져다 쓴다. GPT-2 내부는 입력을 다음 순서로 처리한다.

```
입력 토큰들
     ↓  토큰 임베딩 + 위치 정보(Positional) 추가
각 토큰이 벡터로 변환됨 (순서 정보 포함)
     ↓  Self-Attention 층 (여러 겹 반복)
각 토큰이 문장 속 다른 모든 토큰을 참고해 문맥 반영
     ↓  다음 토큰 확률 계산
가장 그럴듯한 다음 토큰 예측 → 반복(자기회귀)
```

핵심은 Self-Attention이다. LSTM이 글자를 앞에서 뒤로 하나씩 순서대로 처리하는 것과 달리, Self-Attention은 문장 안의 모든 토큰이 서로를 동시에 바라보며 "어떤 단어가 어떤 단어와 관련 깊은지"를 가중치로 계산한다. 덕분에 멀리 떨어진 단어 사이의 관계(장기 의존성)도 잘 포착하고, 순차 처리가 아니라 한 번에 계산하므로 병렬 처리가 가능하다. 이것이 LSTM 대비 Transformer의 가장 큰 구조적 차이다.

| 구성 요소 | 역할 |
|---|---|
| GPT-2 (Transformer 디코더) | Self-Attention으로 모든 토큰 관계를 동시에 계산 → 장기 의존성 포착, 병렬 처리 |
| 사전학습(Pretrained) | 대규모 텍스트로 미리 학습 → 일반 지식 보유 |
| Tokenizer | 문장을 토큰(단어 조각) 숫자로 변환 |
| knowledge_base + retrieve_fact | 외부 사실을 검색해 프롬프트에 주입(하드코딩 미니 버전의 RAG) |

여기서 나온 `knowledge_base`가 딕셔너리 3개짜리 아주 작은 버전인데, Practice_0의 FAISS 벡터 검색이 이걸 제대로 확장한 버전이라는 걸 비교하며 알게 됐다.




---


## 4-5. 주요 코드 분석


문장 생성과 관련된 핵심 코드를 작성하고
각 코드의 역할을 설명하세요.



### 주요 코드


```python


# GPT-2 문장 생성
def generate_text(prompt, max_length=50):
    inputs = tokenizer(prompt, return_tensors="pt")          # 문장 -> 토큰
    outputs = model.generate(
        inputs["input_ids"], attention_mask=inputs["attention_mask"],
        max_length=max_length, do_sample=True, temperature=0.7,
        pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)  # 토큰 -> 문장

# 지식 결합(RAG 스타일)
knowledge_base = {
    "South Korea": "The capital of South Korea is Seoul.",
    "France": "The capital of France is Paris.",
    "Italy": "The capital of Italy is Rome."
}
def retrieve_fact(query):
    for key in knowledge_base:
        if key.lower() in query.lower():
            return knowledge_base[key]
    return "I don't know."
def generate_with_retrieval(query):
    fact = retrieve_fact(query)
    prompt = fact + " " + query          # 진짜 사실을 앞에 붙임
    return generate_text(prompt, 50)
```

**각 코드의 역할**

- `generate_text`: 문장을 토큰으로 바꾸고 GPT-2로 생성한 뒤 다시 문장으로 디코딩한다. `temperature`로 다양성을 조절한다.
- `retrieve_fact`: 질문에 포함된 키워드로 사실을 찾는다. 다만 단순 문자열 매칭이라 의미가 같아도 표현이 다르면 못 찾는 한계가 보였다.
- `generate_with_retrieval`: 찾은 사실을 프롬프트 앞에 붙인다. 모델이 문장을 지어내기 전에 진짜 지식(fact)을 먼저 보게 되는 구조다.


# 5. 세 파일 기술 비교 분석


## 5-1. 파일별 기술 비교


세 파일에서 사용된 방식의 차이점을
비교하여 작성하세요.



| 비교 항목 | practice_0 | practice_1-1 | practice_1-2 |
|---|---|---|---|
| 주요 목적 | 논문 근거 기반 질의응답(RAG) | 문자 단위 LSTM 문장 생성 | Transformer 문장 생성 + 지식 결합 |
| 사용 기술 특징 | LangChain, FAISS, 임베딩, GPT-4o, CoT/SC/ReAct | PyTorch 직접 구현, Embedding+LSTM | 사전학습 GPT-2, Self-Attention, 미니 KB |
| 데이터 처리 방식 | PDF→청크→임베딩→벡터DB→의미검색 | 글자→숫자 인코딩→시퀀스 쌍 | 문장→토큰화 |
| 문장 생성 방식 | 검색 근거 + 프롬프트 → GPT-4o 생성 | 자기회귀 + argmax | 자기회귀 + 샘플링(temperature) |
| 장점 | 환각↓, 근거 기반, 의미 검색 | 원리 이해에 적합, 외부 의존 없음 | 사전학습 지식 보유, 자연스러운 생성 |
| 한계점 | API 비용, 지식원 제한(논문1개) | 데이터 작음, 의미 모름, 다양성 없음 | 환각, KB 하드코딩, 소형 모델 한계 |
| 활용 가능 분야 | 문서 기반 QA, 사내 지식검색 | 교육용 언어모델 원리 학습 | 텍스트 자동완성, 간단 챗봇 |



---


# 5-2. 기술 발전 과정 분석


세 파일의 구현 방식을 비교하여
기술 발전 흐름을 단계적으로 설명하세요.



작성:

세 파일을 **기술 수준 순서**로 다시 배열해보니(파일 번호 순서와는 다르다), 문장 생성 기술의 발전 흐름이 한눈에 들어왔다. 아래 예시의 4단계에 맞춰 정리했다.

```
[기본 생성 방식] Practice_1-1 (LSTM 직접 학습)
   문자 단위로 직접 학습. 순차 처리라 느리고, 장기 의존성이 약하며, 일반 지식이 없다.
        ↓
[학습 기반 생성 방식] Practice_1-2 전반부 (사전학습 GPT-2)
   Self-Attention으로 장기 의존성·병렬 처리를 해결하고, 대규모 사전학습으로 지식을 갖춘다.
   그러나 사실을 지어내는 환각 문제가 남는다.
        ↓
[고도화된 생성 방식] Practice_1-2 후반부 (GPT-2 + 미니 Knowledge Base)
   검색한 사실을 프롬프트에 넣어 환각을 억제한다. 다만 KB가 하드코딩이라 확장성이 없다.
        ↓
[서비스 활용 방식] Practice_0 (FAISS 벡터DB RAG + GPT-4o)
   미니 KB를 실제 벡터 검색으로 확장하고, 의미 기반 검색과 고급 프롬프트(CoT/SC/ReAct)를 더해
   환각을 실질적으로 억제한 서비스 수준 구조에 도달한다.
```

정리하면 **순차 처리(LSTM) → Self-Attention(Transformer) → 검색 기반 지식 보강(RAG)** 으로 발전했다.



---


# 5-3. 각 방식의 개선 방향 비교


각 파일에서 사용된 방식을
개선하기 위한 방법을 작성하세요.



|파일|개선 방향|
|-|-|
|practice_0| 지식원을 논문 1개→다수 문서로 확장, 검색 정확도(chunk·임베딩) 튜닝, 로컬 LLM으로 비용 절감 |
|practice_1-1| 문자→단어/토큰 단위 확대, 데이터셋 확대, LSTM→Transformer 전환, 샘플링으로 다양성 확보 |
|practice_1-2| 하드코딩 KB→실제 벡터DB(RAG)로 교체, 더 크고 정렬된 모델 사용 |



---


# 6. 실제 서비스 적용 방안


각 기술이 실제 AI 서비스에서
어떻게 활용될 수 있는지 설명하세요.



작성:

공부한 내용을 실제 서비스로 연결해보면 다음과 같이 쓸 수 있을 것 같다. 또한 실습에서 다룬 기술은 각각 성격이 달라, 적합한 서비스 영역도 나뉜다.

검색 기반 질의응답 서비스 — Practice_0(RAG) 활용. 사내 규정·논문·제품 매뉴얼처럼 "정확한 근거가 중요한" 문서를 다룰 때 적합하다. 문서를 벡터 DB로 만들어 두면, 직원이 질문했을 때 관련 근거를 찾아 답하므로 환각을 줄일 수 있다. 예: 사내 규정 챗봇, 논문 리서치 보조 도구.
콘텐츠 생성 서비스 — Practice_1-2(사전학습 생성 모델) 활용. 정답이 하나로 정해지지 않고 자연스러운 문장 생성 자체가 목적인 경우에 맞다. 예: 블로그 초안 작성, 이메일·광고 문구 자동완성.
교육 서비스 — Practice_1-1(직접 구현 LSTM) 활용. 완성된 모델을 쓰기보다 원리를 학습하는 것이 목적일 때 적합하다. 학습 데이터·구조를 바꿔가며 결과를 관찰하는 실습 콘텐츠로 쓸 수 있다.
업무 자동화 서비스 — RAG + 에이전트 결합. 검색으로 근거를 확보하고 여러 단계를 자동 수행해야 하는 업무에 적합하다. 예: 자료 조회 → 요약 → 보고서 초안 작성을 한 번에 처리하는 업무 도우미.



---


# 7. 최종 의견


세 가지 방식의 차이점,
장점과 한계,
향후 발전 방향에 대한
종합 의견을 작성하세요.



작성:

세 방식의 차이점은 결국 "지식을 어디서 얻느냐"로 요약된다. LSTM(1-1)은 주어진 데이터만으로 직접 학습하고, GPT-2(1-2)는 대규모 사전학습 지식에 의존하며, RAG(practice_0)는 외부 문서를 실시간으로 검색해 근거로 삼는다.

장점과 한계를 보면, LSTM은 원리를 이해하기엔 좋지만 데이터가 작고 지식이 없어 실용성이 낮다. Transformer는 사전학습 덕에 자연스러운 문장을 만들지만 사실을 지어내는 환각이 남는다. RAG는 근거 기반으로 환각을 크게 줄이지만, 검색 품질과 외부 API 비용에 성능이 좌우된다. 어느 방식도 완벽하지 않고, 목적에 따라 장단이 갈린다는 점이 세 실습을 통해 얻은 가장 큰 결론이다.

향후 발전 방향으로는 (1) 검색을 실제 벡터 DB 기반으로 고도화하고, (2) 최신 지식을 연결해 정보 노후화를 막으며, (3) 사람 의도에 맞게 정렬된(RLHF, 사람 피드백 기반 학습) 더 큰 모델을 쓰고, (4) 에이전트화를 통해 다단계 업무까지 처리하는 방향으로 나아갈 수 있다.

하루 동안의 실습이라 세부까지 완전히 소화하지는 못했지만, "순차 학습 → 어텐션 기반 사전학습 → 검색 기반 지식 보강" 이라는 문장 생성 기술의 큰 발전 흐름은 분명하게 이해할 수 있었다.




---


# 8. 제출 전 확인


아래 항목을 확인하세요.



|확인 항목|확인|
|-|-|
|세 파일 모두 분석 완료|O|
|주요 코드 작성 완료|O|
|코드 역할 설명 완료|O|
|기술 특징 작성 완료|O|
|장점과 한계 작성 완료|O|
|개선 방향 작성 완료|O|
|파일별 비교 작성 완료|O|
|최종 의견 작성 완료|O|

